# Phase 25 (C1d): confirm the mirror window

> `mirrorWindow` (no bridges; each squad sees the other squad's mirror tally on every coupled clue) against C1's `bridge1`, 64 agents, 8 fresh paired seeds (1820-1827), with independent liars and clean: Gate C1d decides whether A1's bridged world replaces bridges with the window (`specs/multiagent/phase25/`).

Artifacts (gitignored): `results/multiagent/colony/c1d/` (`runs.csv`, `turns.jsonl`, `timing.jsonl`, `metrics.json` and their `*_live` copies, `report.md`, `summary.json`). The response cache and the spend ledger are Phase 10's (`results/multiagent/llm/`), under the cap of $250 per calendar month.

- **Main** (`--cell-timeout 0 --flags llm`, `EXPLRL_LLM_LIVE=1`; about $20.35-21.37, approved 2026-11-12): the 32 boards (16 with liars, 16 clean), then `c1dSummary` and the report. Needs `ready25() == []`: `C1D_PLAN` frozen on its pinned hash, and `window_confirm` committed and pushed. Stops and asks at 5% invalid replies or the cap. No pilot: both arms ran at this size, with liars and clean, in C1 and C1b.
- A rerun replays every reply from the cache ($0).

In [ ]:
import json
import os
import shutil
import subprocess
import time

from explRL.bench_grid import writeCSV
from explRL.coop_grid import coopHash
from explRL.llm_budget import PRICES, Budget
from explRL.llm_grid import HAIKU
from explRL.llm_sched import mergeTiming, runWorkload, timingSummary
from explRL.relay_mech import runC1bCell
from explRL.window_confirm import C1D_PLAN, c1dGrid, c1dSummary, writeC1d
from explRL.train import demo_artifact_dir

COLONY, LLM_WORKERS, PINNED = 'multiagent/colony', 32, '27ff916003d1'
LLM = demo_artifact_dir(None, 'multiagent/llm')
LIVE = os.environ.get('EXPLRL_LLM_LIVE') == '1'
LIVE_SPEC = dict(kind='live', cache=str(LLM / 'cache'), cap=float(C1D_PLAN['cap']), ledger=str(LLM / 'spend.jsonl'))


def git(*a):
    try:
        return subprocess.run(['git', *a], capture_output=True, text=True).stdout.strip()
    except OSError:
        return ''


def ready25(pushed=True):
    """Before the run: C1D_PLAN frozen on its pinned hash; with `pushed`, window_confirm committed and HEAD on origin/main."""
    return [m for m, bad in (
        ('C1D_PLAN is not frozen', not C1D_PLAN['frozen']), (f'C1D_PLAN hash is not {PINNED}', coopHash(C1D_PLAN) != PINNED),
        ('window_confirm has uncommitted changes', pushed and bool(git('status', '--porcelain', '--', 'nbs/window_confirm.ipynb', 'explRL/window_confirm.py'))),
        ('HEAD is not pushed to origin/main', pushed and 'origin/main' not in git('branch', '-r', '--contains', 'HEAD'))) if bad]

In [ ]:
# test_ready25 (FR-6): the plan is frozen on the pinned hash; an unfrozen or changed plan blocks the run
assert ready25(pushed=False) == []
C1D_PLAN['frozen'] = False
assert ready25(pushed=False) == ['C1D_PLAN is not frozen', f'C1D_PLAN hash is not {PINNED}']
C1D_PLAN['frozen'] = True
assert ready25(pushed=False) == []

In [ ]:
#| llm
if not LIVE:
    print('skipped: set EXPLRL_LLM_LIVE=1 to make live API calls')
else:
    bad = ready25()
    assert not bad, f'the run waits: {bad}'
    out = demo_artifact_dir('c1d', COLONY)
    logs, cells = out / 'turns', c1dGrid()
    shutil.rmtree(logs, ignore_errors=True)
    t0 = time.time()
    res = runWorkload([(cells[0]['study'], cells)], LIVE_SPEC, LLM_WORKERS, str(logs), run=runC1bCell)
    wall, rows = time.time() - t0, res['rows']
    writeCSV(rows, out / 'runs.csv')
    with open(out / 'turns.jsonl', 'w') as f:
        for p in sorted(logs.glob('*.jsonl')):
            f.write(p.read_text())
    timing = timingSummary(mergeTiming(logs, out / 'timing.jsonl'), wall)
    shutil.rmtree(logs, ignore_errors=True)
    meta = dict(planHash=coopHash(C1D_PLAN), commit=git('rev-parse', '--short', 'HEAD') or 'unknown', model=HAIKU, wall=wall,
                pricesChecked=PRICES[HAIKU]['checked'], cost=float(sum(r['cost'] for r in rows)), calls=int(sum(r['calls'] for r in rows)),
                spent=Budget(LIVE_SPEC['cap'], LIVE_SPEC['ledger']).spent())
    (out / 'metrics.json').write_text(json.dumps(dict(meta, done=res['done'], refused=res['refused'], stopped=res['stopped'], timing=timing),
                                                 indent=2, sort_keys=True, default=str) + '\n')
    for f in ('metrics.json', 'timing.jsonl'):
        if not (out / f.replace('.', '_live.')).exists():
            shutil.copy(out / f, out / f.replace('.', '_live.'))
    print(f"{len(rows)} boards, {meta['calls']} calls, ${meta['cost']:.2f}, ${meta['spent']:.2f} spent this month, {wall:.0f} s; stopped: {res['stopped']}")
    S = c1dSummary(rows)
    assert S['invalid'] < 0.05, f"invalid replies {S['invalid']:.3f}: stop and ask"
    (out / 'summary.json').write_text(json.dumps(dict(meta, summary=S), indent=1, sort_keys=True, default=str) + '\n')
    print(writeC1d(out / 'report.md', S, meta))